In [1]:
# ============================================================
# INSTAGRAM POSTS SCRAPE & PARSE PIPELINE
# ============================================================
#
# PURPOSE:
#   Collect Instagram post data for a list of brand accounts
#   and flatten it into a structured CSV for analysis.
#
# PIPELINE OVERVIEW:
#
#   STEP 0 — Load Accounts
#       Read user_id, username, and brand from
#       user_detailed_table.csv (ig_user_info.csv).
#
#   STEP 1 — Scrape (MODE = "scrape")
#       For each account, call the EnsembleData Instagram
#       API page-by-page and save one raw JSON file per page
#       under: Instagram/posts_info/raw/<user_id>/
#       Errors are saved as separate *_ERROR.json files.
#       Each run appends a summary row to run_log.csv
#       (status, pages fetched, posts fetched, errors).
#
#   STEP 2 — Parse (MODE = "parse")
#       Read all saved raw JSON files recursively, skipping
#       error files. For each post, extract and flatten:
#         - Identity: brand, user ID, username, post ID
#         - Time: post timestamp, scrape timestamp
#         - Media: type (image/video/carousel), dimensions,
#                  cover URL, video URL, audio flag
#         - Engagement: likes, comments, video views
#         - Caption: full text, length, hashtags, mentions
#         - Monetization: paid partnership, affiliate,
#                         co-creators
#         - Context: location, product tags, upcoming events
#       Duplicate posts (by post_id) are dropped.
#       Output: Instagram/ig_post_info.csv
#
# USAGE:
#   Set MODE = "scrape" to collect new data from the API.
#   Set MODE = "parse"  to process already-saved JSON files.
#   Both steps can be run independently.
#
# OUTPUT FILES:
#   Instagram/posts_info/raw/<user_id>/*.json  — raw API pages
#   Instagram/posts_info/run_log.csv           — scrape log
#   Instagram/ig_post_info.csv                 — final table
# ============================================================

import os
import re
import json
import time
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime

import requests
import pandas as pd

In [2]:
# ======================
# Config
# ======================
TOKEN = os.getenv("ENSEMBLEDATA_TOKEN", "vKLIay8miZKARJKU")
ROOT = "https://ensembledata.com/apis"
POSTS_ENDPOINT = "/instagram/user/posts"

RAW_DIR = Path("Instagram/posts_info") / "raw"
PARSED_DIR = Path("Instagram") 
RAW_DIR.mkdir(parents=True, exist_ok=True)
PARSED_DIR.mkdir(parents=True, exist_ok=True)


LOG_PATH = Path("Instagram/posts_info") / "run_log.csv"


In [3]:
# ======================
# Utilities
# ======================
def safe_get(d: Any, path: List[str], default=None):
    cur = d
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def safe_filename(s: str) -> str:
    # keep it OS-safe
    s = s.strip()
    s = re.sub(r"\s+", "_", s)
    return re.sub(r"[^a-zA-Z0-9._-]", "", s)


def now_stamp() -> str:
    # file-friendly timestamp
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def save_json(payload: Dict[str, Any], out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def load_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)
    
    
def append_log_row(row: Dict[str, Any], log_path: Path = LOG_PATH) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)

    df_row = pd.DataFrame([row])

    if log_path.exists():
        df_row.to_csv(log_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(log_path, mode="w", header=True, index=False)


In [4]:
# ======================
# Step 0: Read accounts from user_detailed_table.csv
# ======================
def read_accounts_from_user_detailed_table(csv_path: str) -> pd.DataFrame:
    """
    Must contain: user_id
    Recommended: username, brand (if available)
    """
    df = pd.read_csv(csv_path)
    # normalize columns (case-insensitive)
    df.columns = [c.strip() for c in df.columns]

    if "user_id" not in df.columns:
        raise ValueError("user_detailed_table.csv must contain a 'user_id' column.")

    # Optional columns
    if "username" not in df.columns:
        df["username"] = ""
    if "brand" not in df.columns:
        df["brand"] = ""

    # clean
    df["user_id"] = pd.to_numeric(df["user_id"], errors="coerce")
    df = df.dropna(subset=["user_id"]).copy()
    df["user_id"] = df["user_id"].astype(int)

    df["username"] = df["username"].fillna("").astype(str).str.strip()
    df["brand"] = df["brand"].fillna("").astype(str).str.strip()
    return df

In [5]:
# ======================
# API call (page)
# ======================
def fetch_user_posts_page(
    session: requests.Session,
    user_id: int,
    depth: int = 10,
    chunk_size: int = 10,
    oldest_timestamp: Optional[int] = None,
    start_cursor: str = "",
    alternative_method: bool = False,
    timeout: int = 120,
) -> Dict[str, Any]:
    params = {
        "user_id": int(user_id),
        "depth": int(depth),
        "chunk_size": int(chunk_size),
        "start_cursor": start_cursor,
        "alternative_method": alternative_method,
        "token": TOKEN,
    }
    if oldest_timestamp is not None:
        params["oldest_timestamp"] = int(oldest_timestamp)

    r = session.get(ROOT + POSTS_ENDPOINT, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()


# ======================
# RAW scrape: save JSON per page
# ======================
def scrape_and_save_posts_jsons_for_account(
    session: requests.Session,
    user_id: int,
    username: str = "",
    brand: str = "",
    chunk_size: int = 10,
    depth: int = 10,
    oldest_timestamp: Optional[int] = None,
    max_pages: Optional[int] = 1,      # 1 = only first page; None = all pages
    sleep_seconds: float = 0.2,
) -> List[Path]:
    """
    Saves one JSON file per page into:
        RAW_DIR / <user_id> / <username>_<timestamp>_page<k>.json

    Appends exactly one row per account attempt to LOG_PATH.
    """

    saved_paths: List[Path] = []

    username_clean = safe_filename(username) if username else f"user_{user_id}"
    stamp = now_stamp()

    # per-account raw directory
    account_raw_dir = RAW_DIR / str(user_id)
    account_raw_dir.mkdir(parents=True, exist_ok=True)

    cursor = ""
    page = 0

    # ======================
    # Logging metadata
    # ======================
    start_time = datetime.now()
    pages_fetched = 0
    posts_fetched = 0
    status = "success"
    error_type = ""
    error_message = ""

    while True:
        page += 1

        try:
            payload = fetch_user_posts_page(
                session=session,
                user_id=user_id,
                depth=depth,
                chunk_size=chunk_size,
                oldest_timestamp=oldest_timestamp,
                start_cursor=cursor,
            )

        except requests.HTTPError as e:
            status = "failed"
            error_message = str(e)
            code = e.response.status_code if e.response is not None else None

            if code == 422:
                error_type = "validation_error"
            elif code == 491:
                error_type = "token_not_found"
            elif code == 492:
                error_type = "email_not_verified"
            elif code == 493:
                error_type = "subscription_expired"
            elif code == 495:
                error_type = "daily_quota_exhausted"
            elif code == 500:
                error_type = "server_error"
            else:
                error_type = "http_error"

            err_path = account_raw_dir / f"{username_clean}_{stamp}_page{page}_ERROR.json"
            save_json(
                {
                    "user_id": user_id,
                    "username": username,
                    "brand": brand,
                    "page": page,
                    "cursor": cursor,
                    "http_status": code,
                    "error_type": error_type,
                    "error": error_message,
                },
                err_path,
            )
            saved_paths.append(err_path)
            break

        except Exception as e:
            status = "failed"
            error_type = "unknown"
            error_message = str(e)

            err_path = account_raw_dir / f"{username_clean}_{stamp}_page{page}_ERROR.json"
            save_json(
                {
                    "user_id": user_id,
                    "username": username,
                    "brand": brand,
                    "page": page,
                    "cursor": cursor,
                    "error_type": error_type,
                    "error": error_message,
                },
                err_path,
            )
            saved_paths.append(err_path)
            break

        # ======================
        # Save successful page
        # ======================
        out_path = account_raw_dir / f"{username_clean}_{stamp}_page{page}.json"

        payload["_meta"] = {
            "user_id": user_id,
            "username": username,
            "brand": brand,
            "page": page,
            "scrape_ts": stamp,
            "cursor_used": cursor,
        }

        save_json(payload, out_path)
        saved_paths.append(out_path)

        data_block = payload.get("data", {}) or {}
        posts = data_block.get("posts", []) or []
        next_cursor = data_block.get("last_cursor", "") or ""

        pages_fetched += 1
        posts_fetched += len(posts)

        # stop conditions
        if len(posts) == 0:
            break
        if not next_cursor or next_cursor == cursor:
            break
        if max_pages is not None and page >= max_pages:
            break

        cursor = next_cursor
        time.sleep(sleep_seconds)

    # ======================
    # Finalize & log
    # ======================
    end_time = datetime.now()

    if status == "failed" and pages_fetched > 0:
        status = "partial"

    append_log_row(
        {
            "run_id": stamp,
            "user_id": user_id,
            "username": username,
            "brand": brand,
            "started_at": start_time.isoformat(),
            "ended_at": end_time.isoformat(),
            "status": status,
            "pages_fetched": pages_fetched,
            "posts_fetched": posts_fetched,
            "error_type": error_type,
            "error_message": error_message,
            "raw_files": ";".join([p.name for p in saved_paths]),
        },
        log_path=LOG_PATH,
    )

    return saved_paths



def scrape_posts_from_user_detailed_table(
    user_detailed_table_csv: str = "ig_user_info.csv",
    chunk_size: int = 10,
    depth: int = 10,
    oldest_timestamp: Optional[int] = None,
    max_pages: Optional[int] = 1,
    sleep_seconds: float = 0.2,
) -> None:
    accounts = read_accounts_from_user_detailed_table(user_detailed_table_csv)

    with requests.Session() as session:
        for i, r in accounts.iterrows():
            user_id = int(r["user_id"])
            username = str(r.get("username", "") or "")
            brand = str(r.get("brand", "") or "")

            print(f"[{i+1}/{len(accounts)}] user_id={user_id} username={username} brand={brand}")

            saved = scrape_and_save_posts_jsons_for_account(
                session=session,
                user_id=user_id,
                username=username,
                brand=brand,
                chunk_size=chunk_size,
                depth=depth,
                oldest_timestamp=oldest_timestamp,
                max_pages=max_pages,
                sleep_seconds=sleep_seconds,
            )
            print(f"  -> saved {len(saved)} file(s) into {RAW_DIR}")

In [6]:
# ======================
# PARSE saved JSON pages -> rows
# ======================


HASHTAG_RE = re.compile(r"#(\w+)")
MENTION_RE = re.compile(r"@(\w+)")


def extract_hashtags(text: str) -> str:
    return ", ".join(HASHTAG_RE.findall(text)) if text else ""


def extract_mentions(text: str) -> str:
    return ", ".join(MENTION_RE.findall(text)) if text else ""


def parse_posts_payload_to_rows(posts_payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Flatten ONE page payload to rows (one row per post).
    """
    meta = posts_payload.get("_meta", {}) or {}
    brand_label = meta.get("brand", "")
    meta_username = meta.get("username", "")
    meta_user_id = meta.get("user_id", "")

    rows: List[Dict[str, Any]] = []
    posts = safe_get(posts_payload, ["data", "posts"], []) or []

    for post in posts:
        node = (post.get("node", {}) or {}) if isinstance(post, dict) else {}

        owner = node.get("owner", {}) or {}
        owner_id = owner.get("id", "") or meta_user_id
        owner_username = owner.get("username", "") or meta_username

        post_id = node.get("id", "")
        ts = node.get("taken_at_timestamp", None)
        post_type = node.get("__typename", "")
        shortcode = node.get("shortcode", "")

        readable_date = ts
        #if isinstance(ts, (int, float)) and ts:
        #    readable_date = datetime.fromtimestamp(int(ts)).strftime("%Y-%m-%d %H:%M:%S")
        scrape_ts_str = posts_payload.get("_meta", {}).get("scrape_ts", "")

        scrape_ts_unix = (
            int(datetime.strptime(scrape_ts_str, "%Y%m%d_%H%M%S").timestamp())
            if scrape_ts_str else ""
        )

        cover_url = node.get("thumbnail_src", "")
        cover_width, cover_height = None, None
        thumbs = node.get("thumbnail_resources", [])
        if isinstance(thumbs, list) and thumbs:
            best_thumb = max(thumbs, key=lambda r: r.get("config_width", 0))
            cover_width = best_thumb.get("config_width")
            cover_height = best_thumb.get("config_height")

        likes = safe_get(node, ["edge_media_preview_like", "count"], 0) or 0
        comments = safe_get(node, ["edge_media_to_comment", "count"], 0) or 0

        is_video = bool(node.get("is_video", False))
        video_views = node.get("video_view_count", None) if is_video else None
        video_url = node.get("video_url", "") if is_video else None

        # ---------- Caption ----------
        caption_edges = safe_get(node, ["edge_media_to_caption", "edges"], []) or []
        full_caption = safe_get(caption_edges[0], ["node", "text"], "") if caption_edges else ""
        caption_length = len(full_caption) if full_caption else 0
        hashtags = extract_hashtags(full_caption)
        mentions = extract_mentions(full_caption)
        
        # ---------- Media technical ----------
        #video_duration = node.get("video_duration", None)
        dimensions = node.get("dimensions", {}) or {}
        media_height = dimensions.get("height", None)
        media_width = dimensions.get("width", None)
        has_audio = bool(node.get("has_audio", False))
        like_view_disabled = bool(node.get("like_and_view_counts_disabled", False))
        
        # ---------- Co-creators ----------
        coauthors = node.get("coauthor_producers", []) or []
        cocreators = ", ".join([c.get("username", "") for c in coauthors if isinstance(c, dict)]) if coauthors else ""
        
        # ---------- Monetization ----------
        is_paid = bool(node.get("is_paid_partnership", False))
        is_affiliate = bool(node.get("is_affiliate", False))
        sponsored_partnership = (
            "Paid Partnership" if is_paid else
            ("Affiliate" if is_affiliate else "")
)

        # carousel
        image_count = 0
        video_count = 0
        carousel_media_urls = ""
        if post_type == "GraphSidecar":
            child_edges = safe_get(node, ["edge_sidecar_to_children", "edges"], []) or []
            urls = []
        
            for e in child_edges:
                child = safe_get(e, ["node"], {}) or {}
                child_is_video = bool(child.get("is_video", False))
        
                if child_is_video:
                    video_count += 1
                else:
                    image_count += 1
        
                u = child.get("display_url", "")
                if u:
                    urls.append(u)
        
            carousel_media_urls = ", ".join(urls)
        
        else:
            # single-image or single-video post
            if is_video:
                video_count = 1
                image_count = 0
            else:
                image_count = 1 if node.get("display_url") else 0
                video_count = 0
        
        media_count = image_count + video_count
        
        # ---------- Location & commerce ----------
        location = safe_get(node, ["location", "name"], None)
        has_upcoming_event = bool(node.get("has_upcoming_event", False))
        product_tags = safe_get(node, ["product_tags"], None)
        
        # ---------- Preview ----------
        media_preview = node.get("media_preview", None)
        
        rows.append({
            # ---------- Identity ----------
            "brand": brand_label,
            "ig_brand_ID": owner_id,
            "brand_Username": owner_username,

            "post_id": post_id,
            "product_type": node.get("product_type", ""),
            "post_type": post_type,
            "shortcode": shortcode,

            # ---------- Time ----------
            "timestamp": readable_date,
            "scrape_ts": scrape_ts_unix,
            "page": posts_payload.get("_meta", {}).get("page", ""),
            
            # ---------- Media technical ----------
            "cover_url": cover_url,
            "cover_width": cover_width,
            "cover_height": cover_height,
            "video_url": video_url,
            "media_height": media_height,
            "media_width": media_width,
            #"video_duration": video_duration,
            "has_audio": has_audio,
            #"dash_number_of_qualities": num_qualities,
            #"media_preview": media_preview,
            
            # ---------- Engagement ----------
            "likes": likes,
            "comments": comments,
            "video_views": (video_views if video_views is not None else ""),
            "like_and_view_counts_disabled": like_view_disabled,
            
            # ---------- Media structure ----------
            "image_count": image_count,
            "video_count": video_count,
            "media_count": media_count,
            "carousel_media_urls": carousel_media_urls,

            # ---------- Caption ----------
            "full_caption": full_caption,
            "caption_length": caption_length,
            "hashtags": hashtags,
            "mentions": mentions,
            
            # ---------- Monetization ----------
            "co_creators": cocreators,
            "is_paid_partnership": is_paid,
            "is_affiliate": is_affiliate,
            "sponsored_partnership": sponsored_partnership,
            
            
            # ---------- Context ----------
            "location": location,
            "has_upcoming_event": has_upcoming_event,
            "product_tags": product_tags,
})

    return rows


def parse_saved_posts_jsons(
    raw_dir: Path = RAW_DIR,
    out_csv: str = str(PARSED_DIR / "instagram_posts_parsed.csv"),
) -> pd.DataFrame:
    """
    Recursively reads ALL saved JSON files under:
        RAW_DIR / <user_id> / *.json

    Excludes *ERROR.json files, parses posts into one table,
    and writes a single CSV.
    """

    # 🔹 recursive glob to cover RAW_DIR/<user_id>/*.json
    json_files = sorted(
        [
            p for p in raw_dir.rglob("*.json")
            if "ERROR" not in p.name
        ]
    )

    all_rows: List[Dict[str, Any]] = []

    for p in json_files:
        try:
            payload = load_json(p)
        except Exception as e:
            print(f"[WARN] Failed to load {p}: {e}")
            continue

        rows = parse_posts_payload_to_rows(payload)

        # Optional but recommended: attach provenance
        for r in rows:
            r["_raw_file"] = p.name
            r["_user_id"] = p.parent.name   # folder name = user_id

        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)

    # Deduplicate at post level (pagination overlap protection)
    if not df.empty and "post_id" in df.columns:
        df = (
            df
            .drop_duplicates(subset=["post_id"], keep="first")
            .reset_index(drop=True)
        )

    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False, encoding="utf-8")

    print(
        f"[DONE] Parsed {len(json_files)} JSON file(s) "
        f"-> rows={len(df)} -> {out_csv}"
    )

    return df


In [8]:
# ======================
# MAIN (separated)
# ======================
if __name__ == "__main__":
    # Choose one: "scrape" | "parse"
    MODE = "parse"

    if MODE == "scrape":
        # 1) Read from user_detailed_table.csv (user_id column) and scrape
        scrape_posts_from_user_detailed_table(
            user_detailed_table_csv="/Users/pouriakhansari/Library/CloudStorage/OneDrive-IveyBusinessSchool/Prashant/Content Format/Code/EnsembleData/Instagram/ig_user_info.csv",
            chunk_size=10,
            depth=10,
            oldest_timestamp=None,
            max_pages= 1,       # set None to scrape ALL pages
            sleep_seconds=0.2,
        )

    elif MODE == "parse":
        # 2) Read saved JSON files in posts_info/raw and parse
        df_posts = parse_saved_posts_jsons(
            raw_dir=RAW_DIR,
            out_csv=str(PARSED_DIR / "ig_post_info.csv"),
        )
        df_posts.head(10)

    else:
        raise ValueError("MODE must be either 'scrape' or 'parse'")


[DONE] Parsed 33 JSON file(s) -> rows=3107 -> Instagram/ig_post_info.csv


In [9]:
df_posts

,brand,ig_brand_ID,brand_Username,post_id,product_type,post_type,timestamp,scrape_ts,page,cover_url,...,mentions,co_creators,is_paid_partnership,is_affiliate,sponsored_partnership,location,has_upcoming_event,product_tags,_raw_file,_user_id
0,Google,69550579620,googlegemini,3835422615627999905,clips,GraphVideo,1771438053,1771445983,1,https://instagram.fbkk7-2.fna.fbcdn.net/v/t51....,...,,google,False,False,,None,False,None,google_20260218_151943_page1.json,1067259270
1,Google,1067259270,google,3835405492215196230,clips,GraphVideo,1771436226,1771445983,1,https://instagram.fbkk7-2.fna.fbcdn.net/v/t51....,...,,,False,False,,None,False,None,google_20260218_151943_page1.json,1067259270
2,Google,69550579620,googlegemini,3835391197422670634,,GraphSidecar,1771434281,1771445983,1,https://instagram.fbkk7-2.fna.fbcdn.net/v/t51....,...,,google,False,False,,None,False,None,google_20260218_151943_page1.json,1067259270
3,Google,11421759503,googlefordevs,3834753934065553649,clips,GraphVideo,1771358470,1771445983,1,https://instagram.fbkk7-3.fna.fbcdn.net/v/t51....,...,,"googlegemini, google, googledeepmind",False,False,,None,False,None,google_20260218_151943_page1.json,1067259270
4,Google,1067259270,google,3834664334044237488,,GraphSidecar,1771347632,1771445983,1,https://instagram.fbkk7-3.fna.fbcdn.net/v/t51....,...,,,False,False,,None,False,None,google_20260218_151943_page1.json,1067259270
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3102,adidas,9187952,adidasoriginals,3759770351690102423,,GraphSidecar,1762419574,1770049452,2,https://instagram.flwo7-2.fna.fbcdn.net/v/t51....,...,,,False,False,,None,False,None,adidasoriginals_20260202_112412_page2.json,9187952
3103,adidas,9187952,adidasoriginals,3759253690411160868,clips,GraphVideo,1762358437,1770049452,2,https://instagram.flwo7-2.fna.fbcdn.net/v/t51....,...,,"alexiaelkaim, miaou",False,False,,None,False,None,adidasoriginals_20260202_112412_page2.json,9187952
3104,adidas,9187952,adidasoriginals,3758651514160681654,clips,GraphVideo,1762286500,1770049452,2,https://instagram.flwo7-2.fna.fbcdn.net/v/t51....,...,jharden13,"jharden13, adidasbasketball",False,False,,None,False,None,adidasoriginals_20260202_112412_page2.json,9187952
3105,adidas,9187952,adidasoriginals,3758320715410452818,,GraphSidecar,1762246764,1770049452,2,https://instagram.flwo7-2.fna.fbcdn.net/v/t51....,...,,mrbailey,False,False,,None,False,None,adidasoriginals_20260202_112412_page2.json,9187952
